In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [3]:
pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)

print("profiling pins installed (no vLLM today)")

installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.49.2
profiling pins installed (no vLLM today)


In [4]:
import csv, threading, time

GPU_SAMPLES = "/content/gpu_samples.csv"
_sampler = {"thread": None, "stop": None}

def _sample_loop(stop_event, path, interval_s):
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["t", "util_gpu", "mem_used_mib"])
        t0 = time.time()

        while not stop_event.is_set():
            out = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
            ).stdout.strip()

            parts = [p.strip() for p in out.split(",")]

            if len(parts) == 2:
                w.writerow([
                    round(time.time() - t0, 2),
                    parts[0],
                    parts[1],
                ])
                fh.flush()

            stop_event.wait(interval_s)


def start_sampler(path=GPU_SAMPLES, interval_s=2):
    if _sampler["thread"] and _sampler["thread"].is_alive():
        print("sampler already running; not starting a second one")
        return

    stop = threading.Event()

    th = threading.Thread(
        target=_sample_loop,
        args=(stop, path, interval_s),
        daemon=True,
    )

    th.start()

    _sampler["thread"], _sampler["stop"] = th, stop

    print(f"sampler started -> {path} (every {interval_s}s)")


def stop_sampler():
    if _sampler["stop"]:
        _sampler["stop"].set()

    if _sampler["thread"]:
        _sampler["thread"].join(timeout=5)

    _sampler["thread"], _sampler["stop"] = None, None

    print("sampler stopped")


def read_util_mean(path=GPU_SAMPLES):
    vals = []

    with open(path) as fh:
        for row in csv.DictReader(fh):
            try:
                vals.append(float(row["util_gpu"]))
            except (KeyError, ValueError):
                pass

    return sum(vals) / len(vals) if vals else 0.0

In [5]:
import time
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)


def load(dtype: str):

    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="cuda",
        )

    if dtype == "int8":
        qc = BitsAndBytesConfig(
            load_in_8bit=True
        )

        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=qc,
            device_map="cuda",
        )

    raise ValueError(dtype)


def make_prompt(context_tokens: int) -> str:

    base = (
        "Summarise the following text "
        "in one sentence.\n"
    )

    filler = (
        "The data center runs many small "
        "inference requests all day. " * 400
    )

    ids = tok(
        base + filler
    )["input_ids"][:context_tokens]

    return tok.decode(ids)


def resident_vram_gb() -> float:

    torch.cuda.synchronize()

    return (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )


def profile(
    model,
    dtype: str,
    context: int,
    new_tokens: int = 128,
    batch: int = 1,
):

    prompt = make_prompt(context)

    prompts = [prompt] * batch

    enc = tok(
        prompts,
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    # warm-up: not measured
    _ = model.generate(
        **enc,
        max_new_tokens=8,
        do_sample=False,
    )

    vram = resident_vram_gb()

    start_sampler()

    t0 = time.time()

    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
    )

    dt = time.time() - t0

    stop_sampler()

    gen_tokens = (
        out.shape[1]
        - enc["input_ids"].shape[1]
    ) * batch

    return {
        "dtype": dtype,
        "context": context,
        "vram_gb": round(vram, 3),
        "util_mean": round(
            read_util_mean(),
            1,
        ),
        "tokens_per_s": round(
            gen_tokens / dt,
            1,
        ),
    }


def free_vram():

    gc.collect()
    torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
tok.pad_token = tok.eos_token
tok.padding_side = "left"

In [7]:
print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL)
print("load:", load)
print("profile:", profile)
print("sampler:", start_sampler)

GPU: Tesla T4
Model: Qwen/Qwen2.5-1.5B-Instruct
load: <function load at 0x7997595b3f60>
profile: <function profile at 0x7997595b2a20>
sampler: <function start_sampler at 0x79986f460fe0>


In [8]:
predictions = {
    "fp16_vram_context_512_gb": 3.2,
    "fp16_vram_context_4096_gb": 3.8,
    "single_request_gpu_util_percent": 70,
}

print(predictions)

{'fp16_vram_context_512_gb': 3.2, 'fp16_vram_context_4096_gb': 3.8, 'single_request_gpu_util_percent': 70}


In [9]:
rows = []

for dtype in ["fp16", "int8"]:

    model = load(dtype)

    for context in [
        512,
        2048,
        4096,
    ]:

        row = profile(
            model,
            dtype,
            context,
        )

        print(row)

        rows.append(row)

    del model
    free_vram()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)


...

sampler stopped
{'dtype': 'int8', 'context': 2048, 'vram_gb': 2.035, 'util_mean': 27.3, 'tokens_per_s': 5.7}


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler stopped
{'dtype': 'int8', 'context': 4096, 'vram_gb': 2.309, 'util_mean': 30.9, 'tokens_per_s': 5.3}


In [10]:
model = load("fp16")

b1 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=1,
)

b8 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=8,
)

del model
free_vram()


print("batch 1:", b1)

print("batch 8:", b8)

print(
    "tokens/s ratio:",
    round(
        b8["tokens_per_s"]
        / b1["tokens_per_s"],
        2,
    ),
)

print(
    "util delta:",
    round(
        b8["util_mean"]
        - b1["util_mean"],
        1,
    ),
)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler stopped


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 54.3, 'tokens_per_s': 28.4}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.527, 'util_mean': 80.7, 'tokens_per_s': 214.3}
tokens/s ratio: 7.55
util delta: 26.4


In [11]:
import json

with open(
    "batch_check.json",
    "w",
) as f:

    json.dump(
        {
            "batch1_tokens_per_s":
                b1["tokens_per_s"],

            "batch8_tokens_per_s":
                b8["tokens_per_s"],
        },
        f,
        indent=2,
    )

print("wrote batch_check.json")

wrote batch_check.json


In [13]:
with open(
    "profile.json",
    "w",
) as f:

    json.dump(
        rows,
        f,
        indent=2,
    )

print(
    "wrote",
    len(rows),
    "rows to profile.json",
)

wrote 6 rows to profile.json


In [14]:
!python verify_cell.py

rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 28.4, batch-8 tokens/s: 214.3
GREEN CHECK: PASS


In [15]:
from google.colab import files

for f_ in [
    "profile.json",
    "batch_check.json",
]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



---



---



In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
import subprocess, sys

BITSANDBYTES_PIN = "0.49.2"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [3]:
pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)

print("profiling pins installed")

installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.49.2
profiling pins installed


In [4]:
import torch
import gc

from transformers import AutoModelForCausalLM


MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


def reserved_mb():

    torch.cuda.synchronize()

    return (
        torch.cuda.memory_reserved()
        / (1024 ** 2)
    )


samples = []


for i in range(5):

    model = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="cuda",
        )
    )

    after_load = reserved_mb()

    # Delete the reference in the scope
    # that actually owns it.
    del model

    gc.collect()
    torch.cuda.empty_cache()

    after_unload = reserved_mb()

    row = {
        "cycle": i,
        "after_load_mb":
            round(after_load, 1),
        "after_unload_mb":
            round(after_unload, 1),
    }

    samples.append(row)

    print(row)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'cycle': 0, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 1, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 2, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 3, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 4, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}


In [5]:
model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL,
        torch_dtype=torch.float16,
        device_map="cuda",
    )
)

tok_ids = torch.randint(
    0,
    1000,
    (1, 64),
).to("cuda")


leaked_outputs = []
leak_samples = []


for i in range(20):

    # Intentionally no torch.no_grad()
    out = model(tok_ids)

    # Intentionally keep every output
    leaked_outputs.append(
        out.logits
    )

    row = {
        "iter": i,
        "reserved_mb":
            round(
                reserved_mb(),
                1,
            ),
    }

    leak_samples.append(row)

    if i % 5 == 0:
        print(row)

{'iter': 0, 'reserved_mb': 3254.0}
{'iter': 5, 'reserved_mb': 4294.0}
{'iter': 10, 'reserved_mb': 5354.0}
{'iter': 15, 'reserved_mb': 6414.0}


In [6]:
import numpy as np


def detect_leak(
    samples_mb,
    slope_threshold_mb_per_iter=1.0,
):

    x = np.arange(
        len(samples_mb)
    )

    y = np.array(
        samples_mb
    )

    slope, intercept = np.polyfit(
        x,
        y,
        1,
    )

    leaking = (
        slope
        > slope_threshold_mb_per_iter
    )

    return {
        "slope_mb_per_iter":
            round(float(slope), 3),

        "threshold_mb_per_iter":
            slope_threshold_mb_per_iter,

        "leaking":
            bool(leaking),

        "n_samples":
            len(samples_mb),
    }


leak_result = detect_leak(
    [
        s["reserved_mb"]
        for s in leak_samples
    ]
)


print(leak_result)


assert leak_result["leaking"]

{'slope_mb_per_iter': 211.248, 'threshold_mb_per_iter': 1.0, 'leaking': True, 'n_samples': 20}


In [7]:
del model
del leaked_outputs
del out

gc.collect()
torch.cuda.empty_cache()

print("VRAM after cleanup:", round(reserved_mb(), 1), "MB")

VRAM after cleanup: 24.0 MB


In [8]:
model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL,
        torch_dtype=torch.float16,
        device_map="cuda",
    )
)

In [9]:
fixed_samples = []


with torch.no_grad():

    for i in range(20):

        out = model(tok_ids)

        # Use the result, but DON'T keep
        # the GPU tensor.
        _ = (
            out.logits
            .sum()
            .item()
        )

        row = {
            "iter": i,
            "reserved_mb":
                round(
                    reserved_mb(),
                    1,
                ),
        }

        fixed_samples.append(row)


fixed_result = detect_leak(
    [
        s["reserved_mb"]
        for s in fixed_samples
    ]
)


print(fixed_result)


assert not fixed_result["leaking"]

{'slope_mb_per_iter': 0.286, 'threshold_mb_per_iter': 1.0, 'leaking': False, 'n_samples': 20}


In [10]:
import json


report = {

    "reload_loop_baseline":
        samples,

    "leaky_run":
        leak_result,

    "fixed_run":
        fixed_result,

    "leaky_samples": [
        s["reserved_mb"]
        for s in leak_samples
    ],

    "fixed_samples": [
        s["reserved_mb"]
        for s in fixed_samples
    ],
}


with open(
    "leak_report.json",
    "w",
) as f:

    json.dump(
        report,
        f,
        indent=2,
    )


print(
    json.dumps(
        report,
        indent=2,
    )
)

{
  "reload_loop_baseline": [
    {
      "cycle": 0,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 1,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 2,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 3,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 4,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    }
  ],
  "leaky_run": {
    "slope_mb_per_iter": 211.248,
    "threshold_mb_per_iter": 1.0,
    "leaking": true,
    "n_samples": 20
  },
  "fixed_run": {
    "slope_mb_per_iter": 0.286,
    "threshold_mb_per_iter": 1.0,
    "leaking": false,
    "n_samples": 20
  },
  "leaky_samples": [
    3254.0,
    3462.0,
    3670.0,
    3878.0,
    4086.0,
    4294.0,
    4522.0,
    4730.0,
    4938.0,
    5146.0,
    5354.0,
    5562.0,
    5790.0,
    5998.0,
    6206.0,
    6414.0,
    6622.0,
    6830.0,
    7058.0,
 

In [14]:
!python verify_cell.py

refit slopes agree with the detector; leak reproduced then removed
GREEN CHECK: PASS
